# Retos · Proyectos de simulación

Miniproyectos con varias clases que colaboran: unos juegos deportivos con medallero, un árbol genealógico con validaciones, una red social con *feeds*, un calendario de adviento, un árbol de Navidad con luces, un planificador de objetivos que exporta a texto y una cuadrícula de sensores. Algunos proyectos ocupan dos ejercicios: primero el modelo y después la parte que lo usa. Las entradas se simulan con listas fijas y el azar sale de generadores con semilla. Ideas de [retosdeprogramacion.com](https://retosdeprogramacion.com/ejercicios).

## 1. Juegos deportivos: eventos e inscripciones

Crear una `dataclass` inmutable `Participante` (nombre y país) y una clase `Juegos` que registre eventos (sin repetir nombre) e inscriba participantes en ellos, con `ValueError` si el evento no existe o el participante ya está inscrito. Montar unos Juegos de la Macaronesia y el Atlántico con seis eventos e inscribir a doce deportistas al azar en tres eventos cada uno.

In [1]:
import random
from collections import Counter, defaultdict
from dataclasses import dataclass, field


@dataclass(frozen=True)
class Participante:
    nombre: str
    pais: str


class Juegos:
    def __init__(self, nombre):
        self.nombre = nombre
        self.eventos: dict[str, list[Participante]] = {}
        self.resultados: dict[str, tuple[Participante, ...]] = {}

    def registrar_evento(self, evento):
        if evento in self.eventos:
            raise ValueError(f"el evento {evento!r} ya está registrado")
        self.eventos[evento] = []

    def inscribir(self, evento, participante):
        if evento not in self.eventos:
            raise ValueError(f"no existe el evento {evento!r}")
        if participante in self.eventos[evento]:
            raise ValueError(f"{participante.nombre} ya está inscrito en {evento}")
        self.eventos[evento].append(participante)


juegos = Juegos("Juegos de la Macaronesia y el Atlántico")
for evento in ["100 m lisos", "salto de longitud", "natación 200 m", "vela ligera", "salto del pastor", "lucha"]:
    juegos.registrar_evento(evento)

deportistas = [
    Participante("Aythami", "España"), Participante("Nayra", "España"), Participante("Rúben", "Portugal"),
    Participante("Inês", "Portugal"), Participante("Youssef", "Marruecos"), Participante("Salma", "Marruecos"),
    Participante("Nelson", "Cabo Verde"), Participante("Djeneba", "Cabo Verde"), Participante("Moussa", "Senegal"),
    Participante("Awa", "Senegal"), Participante("Tiago", "Portugal"), Participante("Acerina", "España"),
]
rng = random.Random(1)
for d in deportistas:
    for evento in rng.sample(list(juegos.eventos), 3):
        juegos.inscribir(evento, d)
juegos.registrar_evento("petanca")
juegos.inscribir("petanca", deportistas[0])
juegos.inscribir("petanca", deportistas[4])

for evento, inscritos in juegos.eventos.items():
    print(f"{evento:<18} {len(inscritos):>2} inscritos: {', '.join(p.nombre for p in inscritos)}")

for accion in (lambda: juegos.registrar_evento("lucha"), lambda: juegos.inscribir("petanca", deportistas[0]),
               lambda: juegos.inscribir("surf", deportistas[1])):
    try:
        accion()
    except ValueError as e:
        print("Error:", e)
assert sum(map(len, juegos.eventos.values())) == 12 * 3 + 2

100 m lisos         9 inscritos: Aythami, Nayra, Inês, Youssef, Salma, Nelson, Djeneba, Moussa, Tiago
salto de longitud   6 inscritos: Aythami, Inês, Nelson, Awa, Tiago, Acerina
natación 200 m      3 inscritos: Nayra, Nelson, Djeneba
vela ligera         7 inscritos: Nayra, Rúben, Inês, Youssef, Salma, Awa, Acerina
salto del pastor    6 inscritos: Aythami, Rúben, Youssef, Salma, Djeneba, Moussa
lucha               5 inscritos: Rúben, Moussa, Awa, Tiago, Acerina
petanca             2 inscritos: Aythami, Youssef
Error: el evento 'lucha' ya está registrado
Error: Aythami ya está inscrito en petanca
Error: no existe el evento 'surf'


## 2. Juegos deportivos: resultados y medallero

Escribir `simular(juegos, rng)`, que en cada evento con al menos tres inscritos elija al azar el oro, la plata y el bronce (y avise de los que no se celebran), y `medallero(juegos, clave)`, que cuente las medallas por país o por participante y las ordene por oros, después platas y después bronces. Mostrar ambos rankings y comprobar que se reparten exactamente tres medallas por evento celebrado.

In [2]:
METALES = ("oro", "plata", "bronce")


def simular(juegos, rng):
    for evento, inscritos in juegos.eventos.items():
        if len(inscritos) < 3:
            print(f"{evento}: no se celebra, solo hay {len(inscritos)} inscritos")
            continue
        juegos.resultados[evento] = tuple(rng.sample(inscritos, 3))


def medallero(juegos, clave):
    tabla = defaultdict(lambda: [0, 0, 0])
    for podio in juegos.resultados.values():
        for posicion, p in enumerate(podio):
            tabla[clave(p)][posicion] += 1
    return sorted(tabla.items(), key=lambda kv: ([-m for m in kv[1]], kv[0]))


def imprimir(titulo, filas):
    print(f"\n{titulo:<12} {'oro':>4} {'plata':>6} {'bronce':>7} {'total':>6}")
    for nombre, (o, p, b) in filas:
        print(f"{nombre:<12} {o:>4} {p:>6} {b:>7} {o + p + b:>6}")


simular(juegos, random.Random(2))
for evento, podio in juegos.resultados.items():
    print(f"{evento:<18} " + "  ".join(f"{m}: {p.nombre} ({p.pais})" for m, p in zip(METALES, podio)))
por_pais = medallero(juegos, lambda p: p.pais)
por_deportista = medallero(juegos, lambda p: p.nombre)
imprimir("País", por_pais)
imprimir("Deportista", por_deportista[:5])

celebrados = len(juegos.resultados)
assert celebrados == 6 and "petanca" not in juegos.resultados
for tabla in (por_pais, por_deportista):
    assert [sum(m[i] for _, m in tabla) for i in range(3)] == [celebrados] * 3
    assert [m for _, m in tabla] == sorted((m for _, m in tabla), reverse=True)
assert all(len(set(podio)) == 3 and set(podio) <= set(juegos.eventos[e]) for e, podio in juegos.resultados.items())

petanca: no se celebra, solo hay 2 inscritos
100 m lisos        oro: Aythami (España)  plata: Nayra (España)  bronce: Tiago (Portugal)
salto de longitud  oro: Nelson (Cabo Verde)  plata: Inês (Portugal)  bronce: Acerina (España)
natación 200 m     oro: Nelson (Cabo Verde)  plata: Nayra (España)  bronce: Djeneba (Cabo Verde)
vela ligera        oro: Salma (Marruecos)  plata: Awa (Senegal)  bronce: Rúben (Portugal)
salto del pastor   oro: Salma (Marruecos)  plata: Moussa (Senegal)  bronce: Youssef (Marruecos)
lucha              oro: Acerina (España)  plata: Tiago (Portugal)  bronce: Awa (Senegal)

País          oro  plata  bronce  total
España          2      2       1      5
Cabo Verde      2      0       1      3
Marruecos       2      0       1      3
Portugal        0      2       2      4
Senegal         0      2       1      3

Deportista    oro  plata  bronce  total
Nelson          2      0       0      2
Salma           2      0       0      2
Acerina         1      0       1     

## 3. Árbol genealógico: personas y validaciones

Crear una `dataclass` `Persona` (nombre, año de nacimiento, pareja, progenitores e hijos) y una clase `Arbol` que permita añadir personas, emparejarlas y añadir hijos a una persona (y a su pareja, si la tiene). Lanzar `ErrorGenealogico` cuando alguien quiera ser hijo de sí mismo, tener más de dos progenitores, nacer antes que su progenitor, ser hijo de un descendiente suyo o emparejarse teniendo ya pareja.

In [3]:
class ErrorGenealogico(Exception):
    pass


@dataclass(eq=False)
class Persona:
    nombre: str
    nacimiento: int
    pareja: "Persona | None" = field(default=None, repr=False)
    progenitores: list["Persona"] = field(default_factory=list, repr=False)
    hijos: list["Persona"] = field(default_factory=list, repr=False)


class Arbol:
    def __init__(self):
        self.personas: dict[str, Persona] = {}

    def agregar(self, nombre, nacimiento):
        if nombre in self.personas:
            raise ErrorGenealogico(f"{nombre} ya está en el árbol")
        self.personas[nombre] = Persona(nombre, nacimiento)

    def emparejar(self, a, b):
        pa, pb = self.personas[a], self.personas[b]
        if pa is pb:
            raise ErrorGenealogico(f"{a} no puede emparejarse consigo mismo")
        for p in (pa, pb):
            if p.pareja is not None:
                raise ErrorGenealogico(f"{p.nombre} ya tiene pareja: {p.pareja.nombre}")
        pa.pareja, pb.pareja = pb, pa

    def antepasados(self, p):
        return {a for prog in p.progenitores for a in {prog} | self.antepasados(prog)}

    def agregar_hijo(self, progenitor, hijo):
        pp, ph = self.personas[progenitor], self.personas[hijo]
        nuevos = [pp] + ([pp.pareja] if pp.pareja else [])
        for p in nuevos:
            if p is ph:
                raise ErrorGenealogico(f"{hijo} no puede ser hijo de sí mismo")
            if p in ph.progenitores:
                raise ErrorGenealogico(f"{hijo} ya es hijo de {p.nombre}")
            if ph in self.antepasados(p):
                raise ErrorGenealogico(f"{hijo} es antepasado de {p.nombre}")
            if p.nacimiento >= ph.nacimiento:
                raise ErrorGenealogico(f"{hijo} ({ph.nacimiento}) no puede nacer antes que {p.nombre} ({p.nacimiento})")
        if len(ph.progenitores) + len(nuevos) > 2:
            raise ErrorGenealogico(f"{hijo} ya tiene progenitores: {', '.join(x.nombre for x in ph.progenitores)}")
        for p in nuevos:
            p.hijos.append(ph)
            ph.progenitores.append(p)


arbol = Arbol()
for nombre, año in [("Agustín", 1940), ("Candelaria", 1943), ("Domingo", 1966), ("Nieves", 1968), ("Carmen", 1970),
                    ("Echedey", 1969), ("Yurena", 1972), ("Rayco", 1995), ("Iballa", 1998), ("Dácil", 2001),
                    ("Fayna", 2024), ("Oliver", 1994)]:
    arbol.agregar(nombre, año)
arbol.emparejar("Agustín", "Candelaria")
for hijo in ("Domingo", "Carmen", "Yurena"):
    arbol.agregar_hijo("Agustín", hijo)
arbol.emparejar("Domingo", "Nieves")
arbol.emparejar("Carmen", "Echedey")
arbol.agregar_hijo("Domingo", "Rayco")
arbol.agregar_hijo("Domingo", "Iballa")
arbol.agregar_hijo("Carmen", "Dácil")
arbol.emparejar("Iballa", "Oliver")
arbol.agregar_hijo("Iballa", "Fayna")

pruebas = [
    lambda: arbol.agregar_hijo("Rayco", "Rayco"),
    lambda: arbol.agregar_hijo("Yurena", "Rayco"),
    lambda: arbol.agregar_hijo("Fayna", "Agustín"),
    lambda: arbol.agregar_hijo("Dácil", "Oliver"),
    lambda: arbol.emparejar("Yurena", "Echedey"),
    lambda: arbol.emparejar("Rayco", "Rayco"),
    lambda: arbol.agregar("Dácil", 2003),
]
for prueba in pruebas:
    try:
        prueba()
        print("Sin error (no debería pasar)")
    except ErrorGenealogico as e:
        print("Error:", e)

fayna = arbol.personas["Fayna"]
print("Antepasados de Fayna:", ", ".join(sorted(p.nombre for p in arbol.antepasados(fayna))))
assert {p.nombre for p in arbol.antepasados(fayna)} == {"Iballa", "Oliver", "Domingo", "Nieves", "Agustín", "Candelaria"}
assert all(len(p.progenitores) <= 2 for p in arbol.personas.values())
personas = arbol.personas.values()
assert all(p in h.progenitores for p in personas for h in p.hijos) and all(h in p.hijos for h in personas for p in h.progenitores)

Error: Rayco no puede ser hijo de sí mismo
Error: Rayco ya tiene progenitores: Domingo, Nieves
Error: Agustín es antepasado de Fayna
Error: Oliver (1994) no puede nacer antes que Dácil (2001)
Error: Echedey ya tiene pareja: Carmen
Error: Rayco no puede emparejarse consigo mismo
Error: Dácil ya está en el árbol
Antepasados de Fayna: Agustín, Candelaria, Domingo, Iballa, Nieves, Oliver


## 4. Árbol genealógico: dibujarlo

Añadir una función `dibujar(arbol)` que devuelva las líneas del árbol empezando por las personas sin progenitores (salvo las que entran en la familia como pareja de alguien que sí los tiene): cada línea muestra a la persona y su pareja (`⚭`), y debajo, con `├──`, `└──` y `│`, sus hijos ordenados por año de nacimiento. Comprobar que cada persona aparece exactamente una vez.

In [4]:
import re


def dibujar(arbol):
    lineas = []

    def etiqueta(p):
        texto = f"{p.nombre} ({p.nacimiento})"
        return texto + (f" ⚭ {p.pareja.nombre} ({p.pareja.nacimiento})" if p.pareja else "")

    def rama(p, prefijo, conector, siguiente):
        lineas.append(prefijo + conector + etiqueta(p))
        hijos = set(p.hijos) | set(p.pareja.hijos if p.pareja else [])
        hijos = sorted(hijos, key=lambda h: (h.nacimiento, h.nombre))
        for i, h in enumerate(hijos):
            ultimo = i == len(hijos) - 1
            rama(h, prefijo + siguiente, "└── " if ultimo else "├── ", "    " if ultimo else "│   ")

    vistos = set()
    for p in arbol.personas.values():
        entra_por_pareja = p.pareja is not None and p.pareja.progenitores
        if not p.progenitores and not entra_por_pareja and p not in vistos:
            vistos |= {p, p.pareja}
            rama(p, "", "", "")
    return lineas


dibujo = dibujar(arbol)
print("\n".join(dibujo))
texto = "\n".join(dibujo)
for nombre in arbol.personas:
    assert len(re.findall(rf"\b{nombre}\b", texto)) == 1, nombre

Agustín (1940) ⚭ Candelaria (1943)
├── Domingo (1966) ⚭ Nieves (1968)
│   ├── Rayco (1995)
│   └── Iballa (1998) ⚭ Oliver (1994)
│       └── Fayna (2024)
├── Carmen (1970) ⚭ Echedey (1969)
│   └── Dácil (2001)
└── Yurena (1972)


## 5. Red social: usuarios, publicaciones y «me gusta»

Crear una clase `RedSocial` con usuarios registrados por nombre, que puedan seguir y dejar de seguir a otros (no a sí mismos ni dos veces), publicar textos de como mucho 200 caracteres con su fecha, borrar sus propias publicaciones y dar o quitar «me gusta». Los errores lanzan `ErrorRed`. Cargar una red de ejemplo con fechas fijas.

In [5]:
from datetime import datetime


class ErrorRed(Exception):
    pass


@dataclass(eq=False)
class Publicacion:
    id: int
    autor: str
    texto: str
    fecha: datetime
    me_gusta: set[str] = field(default_factory=set)


class RedSocial:
    MAX_CARACTERES = 200

    def __init__(self):
        self.siguiendo: dict[str, set[str]] = {}
        self.publicaciones: dict[int, Publicacion] = {}
        self._siguiente_id = 1

    def _comprobar(self, *nombres):
        for n in nombres:
            if n not in self.siguiendo:
                raise ErrorRed(f"no existe el usuario {n!r}")

    def registrar(self, nombre):
        if nombre in self.siguiendo:
            raise ErrorRed(f"el usuario {nombre!r} ya existe")
        self.siguiendo[nombre] = set()

    def seguir(self, quien, a_quien):
        self._comprobar(quien, a_quien)
        if quien == a_quien:
            raise ErrorRed("nadie puede seguirse a sí mismo")
        if a_quien in self.siguiendo[quien]:
            raise ErrorRed(f"{quien} ya sigue a {a_quien}")
        self.siguiendo[quien].add(a_quien)

    def dejar_de_seguir(self, quien, a_quien):
        self._comprobar(quien, a_quien)
        if a_quien not in self.siguiendo[quien]:
            raise ErrorRed(f"{quien} no sigue a {a_quien}")
        self.siguiendo[quien].remove(a_quien)

    def seguidores(self, nombre):
        return sorted(u for u, s in self.siguiendo.items() if nombre in s)

    def publicar(self, autor, texto, fecha):
        self._comprobar(autor)
        if not texto.strip() or len(texto) > self.MAX_CARACTERES:
            raise ErrorRed(f"el texto debe tener entre 1 y {self.MAX_CARACTERES} caracteres (tiene {len(texto)})")
        p = Publicacion(self._siguiente_id, autor, texto, fecha)
        self.publicaciones[p.id] = p
        self._siguiente_id += 1
        return p.id

    def borrar(self, autor, id_):
        p = self.publicaciones.get(id_)
        if p is None or p.autor != autor:
            raise ErrorRed(f"{autor} no tiene ninguna publicación {id_}")
        del self.publicaciones[id_]

    def alternar_me_gusta(self, usuario, id_):
        self._comprobar(usuario)
        if id_ not in self.publicaciones:
            raise ErrorRed(f"no existe la publicación {id_}")
        self.publicaciones[id_].me_gusta ^= {usuario}
        return usuario in self.publicaciones[id_].me_gusta


red = RedSocial()
for nombre in ("ayoze", "nauzet", "tara", "cathaysa"):
    red.registrar(nombre)
for a, b in [("ayoze", "nauzet"), ("ayoze", "tara"), ("nauzet", "tara"), ("tara", "ayoze"), ("cathaysa", "ayoze"), ("cathaysa", "tara")]:
    red.seguir(a, b)
red.dejar_de_seguir("ayoze", "nauzet")

mensajes = [
    ("tara", "Subiendo al Roque Nublo con niebla. Ni se ve el roque.", datetime(2026, 11, 2, 8, 30)),
    ("ayoze", "¿Alguien sabe a qué hora sale la guagua de las 7?", datetime(2026, 11, 1, 6, 55)),
    ("nauzet", "Hoy potaje de berros en casa de la abuela.", datetime(2026, 11, 2, 13, 5)),
    ("tara", "Vista desde la cumbre: mar de nubes completo.", datetime(2026, 11, 2, 10, 15)),
    ("ayoze", "Partido el domingo en el campo del barrio, faltan dos.", datetime(2026, 11, 3, 19, 40)),
    ("cathaysa", "Primera vez haciendo surf en Las Américas.", datetime(2026, 11, 1, 17, 0)),
    ("tara", "Bajando. Las piernas ya no son mías.", datetime(2026, 11, 2, 16, 45)),
]
ids = [red.publicar(*m) for m in mensajes]
for usuario, id_ in [("ayoze", 4), ("cathaysa", 4), ("nauzet", 4), ("tara", 5), ("cathaysa", 1), ("cathaysa", 1)]:
    red.alternar_me_gusta(usuario, id_)

pruebas = [
    lambda: red.seguir("tara", "tara"), lambda: red.seguir("tara", "ayoze"), lambda: red.dejar_de_seguir("tara", "nauzet"),
    lambda: red.publicar("nauzet", "a" * 201, datetime(2026, 11, 4)), lambda: red.borrar("nauzet", 1),
    lambda: red.seguir("tara", "gara"),
]
for prueba in pruebas:
    try:
        prueba()
    except ErrorRed as e:
        print("Error:", e)

print("Seguidores de tara:", red.seguidores("tara"))
assert red.seguidores("tara") == ["ayoze", "cathaysa", "nauzet"] and red.siguiendo["ayoze"] == {"tara"}
assert red.publicaciones[4].me_gusta == {"ayoze", "cathaysa", "nauzet"} and red.publicaciones[1].me_gusta == set()
assert red.publicar("nauzet", "b" * 200, datetime(2026, 11, 4)) == 8

Error: nadie puede seguirse a sí mismo
Error: tara ya sigue a ayoze
Error: tara no sigue a nauzet
Error: el texto debe tener entre 1 y 200 caracteres (tiene 201)
Error: nauzet no tiene ninguna publicación 1
Error: no existe el usuario 'gara'
Seguidores de tara: ['ayoze', 'cathaysa', 'nauzet']


## 6. Red social: los dos *feeds*

Añadir `feed_propio(red, usuario, n)` con las `n` publicaciones más recientes del usuario y `feed_seguidos(red, usuario, n)` con las `n` más recientes de las personas a las que sigue, ordenadas de más nueva a más antigua y mostrando autor, fecha, texto y número de «me gusta». Comprobar el orden y que los *feeds* cambian al dejar de seguir o borrar.

In [6]:
def feed_propio(red, usuario, n=10):
    red._comprobar(usuario)
    propias = [p for p in red.publicaciones.values() if p.autor == usuario]
    return sorted(propias, key=lambda p: p.fecha, reverse=True)[:n]


def feed_seguidos(red, usuario, n=10):
    red._comprobar(usuario)
    ajenas = [p for p in red.publicaciones.values() if p.autor in red.siguiendo[usuario]]
    return sorted(ajenas, key=lambda p: p.fecha, reverse=True)[:n]


def mostrar(feed):
    for p in feed:
        print(f"  {p.fecha:%d/%m %H:%M} {p.autor:<9} {p.texto[:50]:<50} ♥ {len(p.me_gusta)}")


print("Feed propio de tara:")
mostrar(feed_propio(red, "tara"))
print("Feed de cathaysa (sigue a ayoze y tara), 4 más recientes:")
feed = feed_seguidos(red, "cathaysa", 4)
mostrar(feed)

assert [p.fecha for p in feed] == sorted((p.fecha for p in feed), reverse=True)
assert {p.autor for p in feed_seguidos(red, "cathaysa", 100)} == {"ayoze", "tara"}
red.dejar_de_seguir("cathaysa", "tara")
red.borrar("ayoze", 5)
print("Tras dejar de seguir a tara y borrar ayoze su publicación 5:")
mostrar(feed_seguidos(red, "cathaysa"))
assert [p.id for p in feed_seguidos(red, "cathaysa")] == [2]

Feed propio de tara:
  02/11 16:45 tara      Bajando. Las piernas ya no son mías.               ♥ 0
  02/11 10:15 tara      Vista desde la cumbre: mar de nubes completo.      ♥ 3
  02/11 08:30 tara      Subiendo al Roque Nublo con niebla. Ni se ve el ro ♥ 0
Feed de cathaysa (sigue a ayoze y tara), 4 más recientes:
  03/11 19:40 ayoze     Partido el domingo en el campo del barrio, faltan  ♥ 1
  02/11 16:45 tara      Bajando. Las piernas ya no son mías.               ♥ 0
  02/11 10:15 tara      Vista desde la cumbre: mar de nubes completo.      ♥ 3
  02/11 08:30 tara      Subiendo al Roque Nublo con niebla. Ni se ve el ro ♥ 0
Tras dejar de seguir a tara y borrar ayoze su publicación 5:
  01/11 06:55 ayoze     ¿Alguien sabe a qué hora sale la guagua de las 7?  ♥ 0


## 7. Calendario de adviento

Dibujar un calendario de 24 casillas en una cuadrícula de 6 columnas, con el número de cada casilla cerrada y `**` en las ya descubiertas. Abrir casillas con una lista fija de elecciones, avisando de las repetidas y de las que no existen, y redibujar al final. Comprobar que las abiertas son exactamente las elecciones válidas sin repetir.

In [7]:
COLUMNAS, CASILLAS = 6, 24


def dibujar_calendario(abiertas):
    lineas = []
    for inicio in range(1, CASILLAS + 1, COLUMNAS):
        dias = range(inicio, inicio + COLUMNAS)
        lineas.append(" ".join("┌────┐" for _ in dias))
        lineas.append(" ".join(f"│ {'**' if d in abiertas else f'{d:02d}'} │" for d in dias))
        lineas.append(" ".join("└────┘" for _ in dias))
    return "\n".join(lineas)


def abrir_casillas(elecciones):
    abiertas = set()
    for e in elecciones:
        if not (isinstance(e, int) and 1 <= e <= CASILLAS):
            print(f"La casilla {e!r} no existe")
        elif e in abiertas:
            print(f"La casilla {e} ya estaba descubierta")
        else:
            abiertas.add(e)
            print(f"Casilla {e} descubierta")
    return abiertas


elecciones = [1, 2, 7, 2, 0, 12, 24, "3", 18, 7, 25, 3]
abiertas = abrir_casillas(elecciones)
calendario = dibujar_calendario(abiertas)
print(calendario)

validas = [e for e in elecciones if isinstance(e, int) and 1 <= e <= CASILLAS]
assert abiertas == set(validas) == {1, 2, 3, 7, 12, 18, 24}
assert calendario.count("**") == len(abiertas) and len(calendario.splitlines()) == 3 * CASILLAS // COLUMNAS
assert all(f" {d:02d} " in calendario for d in range(1, CASILLAS + 1) if d not in abiertas)

Casilla 1 descubierta
Casilla 2 descubierta
Casilla 7 descubierta
La casilla 2 ya estaba descubierta
La casilla 0 no existe
Casilla 12 descubierta
Casilla 24 descubierta
La casilla '3' no existe
Casilla 18 descubierta
La casilla 7 ya estaba descubierta
La casilla 25 no existe
Casilla 3 descubierta
┌────┐ ┌────┐ ┌────┐ ┌────┐ ┌────┐ ┌────┐
│ ** │ │ ** │ │ ** │ │ 04 │ │ 05 │ │ 06 │
└────┘ └────┘ └────┘ └────┘ └────┘ └────┘
┌────┐ ┌────┐ ┌────┐ ┌────┐ ┌────┐ ┌────┐
│ ** │ │ 08 │ │ 09 │ │ 10 │ │ 11 │ │ ** │
└────┘ └────┘ └────┘ └────┘ └────┘ └────┘
┌────┐ ┌────┐ ┌────┐ ┌────┐ ┌────┐ ┌────┐
│ 13 │ │ 14 │ │ 15 │ │ 16 │ │ 17 │ │ ** │
└────┘ └────┘ └────┘ └────┘ └────┘ └────┘
┌────┐ ┌────┐ ┌────┐ ┌────┐ ┌────┐ ┌────┐
│ 19 │ │ 20 │ │ 21 │ │ 22 │ │ 23 │ │ ** │
└────┘ └────┘ └────┘ └────┘ └────┘ └────┘


## 8. Árbol de Navidad

Crear una clase `ArbolNavidad` de altura configurable: una estrella `@` arriba, filas de hojas `*` cada vez más anchas (la fila $i$ tiene $2i+1$) y un tronco `|||`. Permitir poner y quitar bolas (`o`) y luces en huecos libres al azar, sin que se solapen, y encender y apagar las luces (encendidas se ven como `+`, apagadas como hojas). Lanzar `ValueError` si no quedan huecos o no hay tantas como se quieren quitar, y comprobar que el árbol tiene $h^2$ huecos.

In [8]:
class ArbolNavidad:
    def __init__(self, altura, rng):
        if altura < 2:
            raise ValueError("la altura mínima es 2")
        self.altura, self.rng = altura, rng
        self.huecos = [(f, c) for f in range(altura) for c in range(altura - 1 - f, altura + f)]
        self.adornos = {"bolas": set(), "luces": set()}
        self.encendidas = True

    def libres(self):
        ocupados = self.adornos["bolas"] | self.adornos["luces"]
        return [h for h in self.huecos if h not in ocupados]

    def poner(self, tipo, n):
        libres = self.libres()
        if n > len(libres):
            raise ValueError(f"no caben {n} {tipo}: solo quedan {len(libres)} huecos")
        self.adornos[tipo].update(self.rng.sample(libres, n))

    def quitar(self, tipo, n):
        puestos = sorted(self.adornos[tipo])
        if n > len(puestos):
            raise ValueError(f"no se pueden quitar {n} {tipo}: solo hay {len(puestos)}")
        self.adornos[tipo].difference_update(self.rng.sample(puestos, n))

    def interruptor(self):
        self.encendidas = not self.encendidas

    def dibujar(self):
        ancho = 2 * self.altura - 1
        filas = [[" "] * ancho for _ in range(self.altura)]
        for f, c in self.huecos:
            if (f, c) in self.adornos["bolas"]:
                filas[f][c] = "o"
            elif (f, c) in self.adornos["luces"] and self.encendidas:
                filas[f][c] = "+"
            else:
                filas[f][c] = "*"
        tronco = " " * (self.altura - 2) + "|||"
        return "\n".join([" " * (self.altura - 1) + "@", *("".join(f).rstrip() for f in filas), tronco, tronco])


arbol_navidad = ArbolNavidad(7, random.Random(8))
arbol_navidad.poner("bolas", 8)
arbol_navidad.poner("luces", 10)
print(arbol_navidad.dibujar())
arbol_navidad.interruptor()
arbol_navidad.quitar("bolas", 3)
print("\nLuces apagadas y tres bolas menos:")
print(arbol_navidad.dibujar())

for accion in (lambda: arbol_navidad.poner("bolas", 100), lambda: arbol_navidad.quitar("luces", 11), lambda: ArbolNavidad(1, None)):
    try:
        accion()
    except ValueError as e:
        print("Error:", e)

for h in range(2, 15):
    a = ArbolNavidad(h, random.Random(h))
    assert len(a.huecos) == len(set(a.huecos)) == h * h
    a.poner("bolas", h)
    a.poner("luces", len(a.libres()) // 2)
    assert not a.adornos["bolas"] & a.adornos["luces"]
    dibujo = a.dibujar()
    assert dibujo.count("o") == h and dibujo.count("+") == len(a.adornos["luces"])
    a.interruptor()
    assert a.dibujar().count("+") == 0 and a.dibujar().count("*") == h * h - h

      @
      *
     +o*
    *o**o
   ***o*o*
  **+*+**oo
 ******++***
+*++*++**o***
     |||
     |||

Luces apagadas y tres bolas menos:
      @
      *
     *o*
    *o**o
   *****o*
  *******o*
 ***********
*************
     |||
     |||
Error: no caben 100 bolas: solo quedan 34 huecos
Error: no se pueden quitar 11 luces: solo hay 10
Error: la altura mínima es 2


## 9. Planificador de objetivos: el plan mensual

Crear una `dataclass` `Objetivo` (meta, cantidad, unidad y número de meses, de 1 a 12) con un método `reparto()` que divida la cantidad entre los meses de la forma más uniforme posible (los primeros meses reciben el resto), y una clase `Planificador` con como mucho 10 objetivos que construya el plan de cada mes. Comprobar que cada reparto suma la cantidad y que sus partes difieren como mucho en 1.

In [9]:
MESES = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio", "Agosto", "Septiembre", "Octubre",
         "Noviembre", "Diciembre"]


@dataclass
class Objetivo:
    meta: str
    cantidad: int
    unidad: str
    meses: int = 12

    def __post_init__(self):
        if self.cantidad <= 0 or not 1 <= self.meses <= 12:
            raise ValueError(f"objetivo no válido: {self.cantidad} {self.unidad} en {self.meses} meses")

    def reparto(self):
        base, resto = divmod(self.cantidad, self.meses)
        return [base + (i < resto) for i in range(self.meses)]


class Planificador:
    MAX_OBJETIVOS = 10

    def __init__(self, anio):
        self.anio = anio
        self.objetivos: list[Objetivo] = []

    def agregar(self, objetivo):
        if len(self.objetivos) == self.MAX_OBJETIVOS:
            raise ValueError(f"ya hay {self.MAX_OBJETIVOS} objetivos: no caben más")
        self.objetivos.append(objetivo)

    def plan(self):
        meses = {m: [] for m in MESES}
        for n, obj in enumerate(self.objetivos, 1):
            for mes, cantidad in zip(MESES, obj.reparto()):
                if cantidad:
                    meses[mes].append(f"[ ] {n}. {obj.meta} ({cantidad} de {obj.cantidad} {obj.unidad})")
        return meses


planificador = Planificador(2027)
for obj in [Objetivo("Leer libros", 20, "libros"), Objetivo("Correr", 600, "km"),
            Objetivo("Subir picos de las islas", 7, "picos", 7), Objetivo("Ahorrar", 1500, "€", 10),
            Objetivo("Aprender a tocar el timple", 3, "canciones", 6)]:
    planificador.agregar(obj)

for mes, tareas in list(planificador.plan().items())[:3]:
    print(f"{mes}:")
    print("\n".join("  " + t for t in tareas))
print("…")

for i in range(5, 10):
    planificador.agregar(Objetivo(f"Objetivo extra {i + 1}", 12, "veces"))
try:
    planificador.agregar(Objetivo("Uno de más", 1, "vez"))
except ValueError as e:
    print("Error:", e)
try:
    Objetivo("Imposible", 10, "veces", 13)
except ValueError as e:
    print("Error:", e)

random.seed(9)
for _ in range(2000):
    o = Objetivo("prueba", random.randint(1, 500), "u", random.randint(1, 12))
    r = o.reparto()
    assert sum(r) == o.cantidad and len(r) == o.meses and max(r) - min(r) <= 1 and r == sorted(r, reverse=True)

Enero:
  [ ] 1. Leer libros (2 de 20 libros)
  [ ] 2. Correr (50 de 600 km)
  [ ] 3. Subir picos de las islas (1 de 7 picos)
  [ ] 4. Ahorrar (150 de 1500 €)
  [ ] 5. Aprender a tocar el timple (1 de 3 canciones)
Febrero:
  [ ] 1. Leer libros (2 de 20 libros)
  [ ] 2. Correr (50 de 600 km)
  [ ] 3. Subir picos de las islas (1 de 7 picos)
  [ ] 4. Ahorrar (150 de 1500 €)
  [ ] 5. Aprender a tocar el timple (1 de 3 canciones)
Marzo:
  [ ] 1. Leer libros (2 de 20 libros)
  [ ] 2. Correr (50 de 600 km)
  [ ] 3. Subir picos de las islas (1 de 7 picos)
  [ ] 4. Ahorrar (150 de 1500 €)
  [ ] 5. Aprender a tocar el timple (1 de 3 canciones)
…
Error: ya hay 10 objetivos: no caben más
Error: objetivo no válido: 10 veces en 13 meses


## 10. Planificador de objetivos: exportar a texto

Añadir `exportar(planificador, ruta)`, que escriba el plan en un fichero `.txt` con un título, los objetivos con su meta total y cada mes con sus tareas (omitiendo los meses vacíos). Leer el fichero y comprobar que contiene una tarea por cada mes en que cada objetivo tiene algo que hacer.

In [10]:
from pathlib import Path


def exportar(planificador, ruta):
    lineas = [f"Objetivos para {planificador.anio}", ""]
    lineas += [f"{n}. {o.meta}: {o.cantidad} {o.unidad} en {o.meses} meses" for n, o in enumerate(planificador.objetivos, 1)]
    for mes, tareas in planificador.plan().items():
        if tareas:
            lineas += ["", f"{mes}:"] + [f"  {t}" for t in tareas]
    ruta = Path(ruta)
    ruta.write_text("\n".join(lineas) + "\n", encoding="utf-8")
    return ruta


ruta = exportar(planificador, "objetivos_2027.txt")
contenido = ruta.read_text(encoding="utf-8")
print("\n".join(contenido.splitlines()[:22]))
print(f"… ({len(contenido.splitlines())} líneas en {ruta.name})")

tareas = [l for l in contenido.splitlines() if l.strip().startswith("[ ]")]
assert len(tareas) == sum(sum(1 for x in o.reparto() if x) for o in planificador.objetivos)
assert sum(l.endswith(":") and l[:-1] in MESES for l in contenido.splitlines()) == 12

Objetivos para 2027

1. Leer libros: 20 libros en 12 meses
2. Correr: 600 km en 12 meses
3. Subir picos de las islas: 7 picos en 7 meses
4. Ahorrar: 1500 € en 10 meses
5. Aprender a tocar el timple: 3 canciones en 6 meses
6. Objetivo extra 6: 12 veces en 12 meses
7. Objetivo extra 7: 12 veces en 12 meses
8. Objetivo extra 8: 12 veces en 12 meses
9. Objetivo extra 9: 12 veces en 12 meses
10. Objetivo extra 10: 12 veces en 12 meses

Enero:
  [ ] 1. Leer libros (2 de 20 libros)
  [ ] 2. Correr (50 de 600 km)
  [ ] 3. Subir picos de las islas (1 de 7 picos)
  [ ] 4. Ahorrar (150 de 1500 €)
  [ ] 5. Aprender a tocar el timple (1 de 3 canciones)
  [ ] 6. Objetivo extra 6 (1 de 12 veces)
  [ ] 7. Objetivo extra 7 (1 de 12 veces)
  [ ] 8. Objetivo extra 8 (1 de 12 veces)
… (140 líneas en objetivos_2027.txt)


## 11. Sensores: la zona crítica

Una cuadrícula de 20 × 20 sensores mide valores enteros de 0 a 99 (generados con `numpy` y semilla fija). Encontrar la zona de 3 × 3 con mayor suma usando una tabla de sumas acumuladas (cada suma de zona sale con cuatro accesos) y comprobarla con fuerza bruta y con `sliding_window_view`. Dibujar la cuadrícula marcando la zona.

In [11]:
import math

import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

N, K = 20, 3
lecturas = np.random.default_rng(11).integers(0, 100, size=(N, N))

acumulada = np.zeros((N + 1, N + 1), dtype=int)
acumulada[1:, 1:] = lecturas.cumsum(axis=0).cumsum(axis=1)
sumas = acumulada[K:, K:] - acumulada[:-K, K:] - acumulada[K:, :-K] + acumulada[:-K, :-K]
f0, c0 = map(int, np.unravel_index(sumas.argmax(), sumas.shape))
print(f"Zona crítica: filas {f0}–{f0 + K - 1}, columnas {c0}–{c0 + K - 1}, suma {sumas[f0, c0]}")

for f in range(N):
    print("".join(f"[{v:2d}]" if f0 <= f < f0 + K and c0 <= c < c0 + K else f" {v:2d} " for c, v in enumerate(lecturas[f])))

fuerza_bruta = max((int(lecturas[i:i + K, j:j + K].sum()), i, j) for i in range(N - K + 1) for j in range(N - K + 1))
assert fuerza_bruta[0] == sumas.max()
assert np.array_equal(sliding_window_view(lecturas, (K, K)).sum(axis=(2, 3)), sumas)

Zona crítica: filas 0–2, columnas 16–18, suma 794
 13  12  79  49  59  60  71   2  48  14  40  92  54   7  54  12 [75][94][97] 62 
 86  36  14  51  44  66  99  27  85  13  34  78  24  67  45  51 [94][81][83] 54 
 98  98  13  20  30  55  82  48  98  35  92  59  72  23  59  80 [88][86][96] 12 
 77  46  68  27   1   8  97  89  30  42  24  14  85  67   7  20  56  90  99  21 
 61   3  17  20  44  34  72  46  33  90  62  69  74  33  89   1  30  15   0  99 
  8  45  81  69  85   5  49   3  19  84   7  58   2  30  93  31  30   8  77  17 
 49   2   2  83   0  46  42  12  64  73  98  19  92   6  14  59  86  89  73   2 
 41  80  52  19  68   9  69   1  36  29  93  72  65  49  28  85  94  21  24  31 
 47  25  40  97  36  94  50  34  93  43  77  31  99  74  87   4  10   6  53  40 
 97  24  16  84  30  74  28  54   8  66  91  69  87  78  56  92  47  14  55  62 
 17  14  66  44  48  78  68  89  16  75  54   3   7  35  63  16  54  99  14  14 
 96  24  87  35  75   6  63  87  33  63  21  15   0  49  52

## 12. Sensores: umbral y distancias

Listar los sensores cuya lectura supera un umbral de 97 y calcular la distancia del centro de la zona crítica a la estación base situada en la esquina `(0, 0)` y al sensor por encima del umbral más cercano, en línea recta (euclídea) y en número de pasos por la cuadrícula (Manhattan). Comprobar las distancias con `math.dist` y `numpy.hypot`.

In [12]:
UMBRAL = 97
altos = [tuple(map(int, p)) for p in np.argwhere(lecturas > UMBRAL)]
print(f"{len(altos)} sensores por encima de {UMBRAL}:", ", ".join(f"{p}={lecturas[p]}" for p in altos))
assert altos == [(f, c) for f in range(N) for c in range(N) if lecturas[f, c] > UMBRAL]

centro = (f0 + 1, c0 + 1)


def distancias(a, b):
    return math.dist(a, b), abs(a[0] - b[0]) + abs(a[1] - b[1])


euclidea, manhattan = distancias(centro, (0, 0))
print(f"Centro de la zona crítica {centro} → estación (0, 0): {euclidea:.2f} en línea recta, {manhattan} pasos")
cercano = min(altos, key=lambda p: distancias(centro, p))
euclidea_c, manhattan_c = distancias(centro, cercano)
print(f"Sensor alto más cercano: {cercano}, a {euclidea_c:.2f} en línea recta y {manhattan_c} pasos")

assert math.isclose(euclidea, np.hypot(*centro)) and manhattan == sum(centro)
assert all(distancias(centro, p) >= distancias(centro, cercano) for p in altos)

9 sensores por encima de 97: (1, 6)=99, (2, 0)=98, (2, 1)=98, (2, 8)=98, (3, 18)=99, (4, 19)=99, (6, 10)=98, (8, 12)=99, (10, 17)=99
Centro de la zona crítica (1, 17) → estación (0, 0): 17.03 en línea recta, 18 pasos
Sensor alto más cercano: (3, 18), a 2.24 en línea recta y 3 pasos
